### Multi Relation Embeddings: Demo using TransE Enbeddings in the PyKEEN Framework

In [2]:
#conda deactivate
#conda env remove -n kge -y
#conda create -n kge -y python=3.11 pip
#conda activate kge
#conda install -y -c pytorch pytorch torchvision torchaudio
#conda install -y -c conda-forge jupyterlab ipykernel numpy pandas scikit-learn
#python -m pip install --upgrade pip
#python -m pip install "pykeen>=1.11.0"
#python -m ipykernel install --user --name kge --display-name "Python (kge)"
#python -c "import torch, torchvision; from torchvision import extension; import pykeen; from pykeen.models import TransE; print(torch.__version__, torchvision.__version__, extension._has_ops(), pykeen.__version__)"

torch: 2.4.1 /opt/anaconda3/lib/python3.12/site-packages/torch/__init__.py
torchvision: 0.19.1 /opt/anaconda3/lib/python3.12/site-packages/torchvision/__init__.py
extension ok; has ops: True


In [12]:
# =========================
# PyKEEN TransE 
# =========================
# Install:
#   pip install pykeen torch

import torch
from pykeen.triples import TriplesFactory
from pykeen.models import TransE
from pykeen.training import SLCWATrainingLoop
from pykeen.evaluation import RankBasedEvaluator

import random
import numpy as np
from pykeen.triples import TriplesFactory
from pykeen.models import TransE
from pykeen.training import SLCWATrainingLoop

In [ ]:
# -------------------------
# A. CareTrace-domain mini KG (triples)
# -------------------------
triples = [
    # symptoms -> diagnoses
    ("SYM:fever", "suggests_dx", "DX:influenza"),
    ("SYM:cough", "suggests_dx", "DX:influenza"),
    ("SYM:myalgia", "suggests_dx", "DX:influenza"),
    ("SYM:shortness_of_breath", "suggests_dx", "DX:pneumonia"),
    ("SYM:cough", "suggests_dx", "DX:pneumonia"),
    ("SYM:chest_pain", "suggests_dx", "DX:pneumonia"),
    ("SYM:polyuria", "suggests_dx", "DX:diabetes"),
    ("SYM:polydipsia", "suggests_dx", "DX:diabetes"),
    ("SYM:weight_loss", "suggests_dx", "DX:diabetes"),
    ("SYM:fever", "suggests_dx", "DX:meningitis"),
    ("SYM:neck_stiffness", "suggests_dx", "DX:meningitis"),
    ("SYM:altered_mental_status", "suggests_dx", "DX:meningitis"),

    # diagnoses -> tests
    ("DX:influenza", "requires_test", "TEST:rapid_flu"),
    ("DX:pneumonia", "requires_test", "TEST:chest_xray"),
    ("DX:pneumonia", "requires_test", "TEST:cbc"),
    ("DX:diabetes", "requires_test", "TEST:hba1c"),
    ("DX:diabetes", "requires_test", "TEST:fasting_glucose"),
    ("DX:meningitis", "requires_test", "TEST:lumbar_puncture"),

    # diagnoses -> meds
    ("DX:influenza", "treated_by", "DRUG:oseltamivir"),
    ("DX:influenza", "treated_by", "DRUG:acetaminophen"),
    ("DX:pneumonia", "treated_by", "DRUG:amoxicillin"),
    ("DX:pneumonia", "treated_by", "DRUG:azithromycin"),
    ("DX:diabetes", "treated_by", "DRUG:metformin"),
    ("DX:diabetes", "treated_by", "DRUG:insulin"),
    ("DX:meningitis", "treated_by", "DRUG:ceftriaxone"),

    # contraindications / interactions
    ("DRUG:metformin", "contraindicated_with", "COND:renal_failure"),
    ("DRUG:azithromycin", "contraindicated_with", "COND:qt_prolongation"),
    ("DRUG:azithromycin", "interacts_with", "DRUG:warfarin"),
]

In [ ]:
# -------------------------
# TransE embeddings model
# -------------------------

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# --- build triples factory ---
triples_np = np.array(triples, dtype=str)
tf = TriplesFactory.from_labeled_triples(triples_np)

# --- train on all triples (simple demo setting) ---
train_tf = tf
test_tf = tf

model = TransE(triples_factory=train_tf, embedding_dim=64, scoring_fct_norm=1)
optimizer = torch.optim.Adam(model.get_grad_params(), lr=1e-2)

training_loop = SLCWATrainingLoop(model=model, triples_factory=train_tf, optimizer=optimizer)
losses = training_loop.train(triples_factory=train_tf, num_epochs=200, batch_size=256)
print("done training; last loss:", float(losses[-1]))


In [ ]:
# -------------------------
# Quick filtered evaluation 
# -------------------------
evaluator = RankBasedEvaluator(filtered=True)
results = evaluator.evaluate(model=model, mapped_triples=test_tf.mapped_triples, additional_filter_triples=[train_tf.mapped_triples])
print("Filtered MRR:", float(results.get_metric("mrr")))

In [ ]:
# -------------------------
# Ranking helper: (head, relation, ?) -> top tails
# -------------------------
entity_to_id = train_tf.entity_to_id
relation_to_id = train_tf.relation_to_id
id_to_entity = {v: k for k, v in entity_to_id.items()}

def rank_tails(head_label: str, relation_label: str, topk: int = 8):
    h = entity_to_id[head_label]
    r = relation_to_id[relation_label]
    hr = torch.tensor([[h, r]], dtype=torch.long)
    with torch.no_grad():
        scores = model.score_t(hr_batch=hr).view(-1)
    top_ids = torch.topk(scores, k=min(topk, scores.numel())).indices.tolist()
    return [(id_to_entity[i], float(scores[i])) for i in top_ids]

In [10]:
# -------------------------
# "Soft reasoning" candidate proposals
# -------------------------
print("\nPropose diagnoses for a symptom:")
for sym in ["SYM:fever", "SYM:cough", "SYM:neck_stiffness"]:
    cands = [(e, s) for e, s in rank_tails(sym, "suggests_dx", topk=10) if e.startswith("DX:")]
    print(f"\n({sym}, suggests_dx, ?)")
    for e, s in cands[:5]:
        print(f"  {e:18s} score={s:.3f}")

print("\nPropose tests for a diagnosis:")
for dx in ["DX:pneumonia", "DX:diabetes", "DX:meningitis"]:
    cands = [(e, s) for e, s in rank_tails(dx, "requires_test", topk=10) if e.startswith("TEST:")]
    print(f"\n({dx}, requires_test, ?)")
    for e, s in cands[:5]:
        print(f"  {e:20s} score={s:.3f}")

print("\nPropose treatments for a diagnosis:")
for dx in ["DX:influenza", "DX:pneumonia", "DX:diabetes"]:
    cands = [(e, s) for e, s in rank_tails(dx, "treated_by", topk=10) if e.startswith("DRUG:")]
    print(f"\n({dx}, treated_by, ?)")
    for e, s in cands[:5]:
        print(f"  {e:20s} score={s:.3f}")

No random seed is specified. This may lead to non-reproducible results.
Training epochs on cpu:   0%|                        | 0/200 [00:00<?, ?epoch/s]
Training batches on cpu:   0%|                   | 0.00/1.00 [00:00<?, ?batch/s]
Training epochs on cpu:   0%| | 0/200 [00:00<?, ?epoch/s, loss=1.26, prev_loss=n
Training batches on cpu:   0%|                   | 0.00/1.00 [00:00<?, ?batch/s]
Training epochs on cpu:   1%| | 2/200 [00:00<00:17, 11.44epoch/s, loss=0.836, pr
Training batches on cpu:   0%|                   | 0.00/1.00 [00:00<?, ?batch/s]
Training epochs on cpu:   1%| | 2/200 [00:00<00:17, 11.44epoch/s, loss=0.855, pr
Training batches on cpu:   0%|                   | 0.00/1.00 [00:00<?, ?batch/s]
Training epochs on cpu:   2%| | 4/200 [00:00<00:17, 11.47epoch/s, loss=0.474, pr
Training batches on cpu:   0%|                   | 0.00/1.00 [00:00<?, ?batch/s]
Training epochs on cpu:   2%| | 4/200 [00:00<00:17, 11.47epoch/s, loss=0.512, pr
Training batches on cpu:   0%|       

done training; last loss: 0.16925548017024994


Evaluating on cpu:   0%|                        | 0.00/28.0 [00:00<?, ?triple/s]Encountered tensors on device_types={'cpu'} while only ['cuda'] are considered safe for automatic memory utilization maximization. This may lead to undocumented crashes (but can be safe, too).
Evaluating on cpu: 100%|████████████████| 28.0/28.0 [00:00<00:00, 1.08ktriple/s]

Filtered MRR: 0.9751983880996704

Propose diagnoses for a symptom:

(SYM:fever, suggests_dx, ?)
  DX:meningitis      score=-9.579
  DX:influenza       score=-10.756
  DX:pneumonia       score=-11.700
  DX:diabetes        score=-11.869

(SYM:cough, suggests_dx, ?)
  DX:pneumonia       score=-7.043
  DX:influenza       score=-7.301
  DX:diabetes        score=-10.390
  DX:meningitis      score=-10.397

(SYM:neck_stiffness, suggests_dx, ?)
  DX:meningitis      score=-6.492
  DX:influenza       score=-9.066
  DX:diabetes        score=-9.591

Propose tests for a diagnosis:

(DX:pneumonia, requires_test, ?)
  TEST:chest_xray      score=-6.555
  TEST:cbc             score=-8.056
  TEST:rapid_flu       score=-9.529
  TEST:lumbar_puncture score=-10.908

(DX:diabetes, requires_test, ?)
  TEST:hba1c           score=-7.065
  TEST:fasting_glucose score=-7.658
  TEST:lumbar_puncture score=-9.949
  TEST:chest_xray      score=-10.458
  TEST:cbc             score=-10.514

(DX:meningitis, requires_test, 